
# Name: Lucky Singh
# DataAnalytics-L1- Cleaning Data


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [10]:
df = pd.read_csv("data/Titanic-Dataset.csv")

In [11]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Dataset Information

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


## Missing Value Handling

In [13]:
df["Age"] = df["Age"].fillna(df["Age"].median())

In [14]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

In [15]:
df = df.drop(columns=["Cabin"])

In [16]:
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## Duplicate Row Removal

In [17]:
duplicates = df.duplicated().sum()
print("Duplicate Rows Before:", duplicates)

Duplicate Rows Before: 0


In [18]:
df = df.drop_duplicates()

In [19]:
print("Duplicate Rows After:", df.duplicated().sum())

Duplicate Rows After: 0


## Data Standardization

In [20]:
df["Sex"] = df["Sex"].replace({
    "male": "Male",
    "female": "Female"
})

In [21]:
df["Name"] = df["Name"].str.title().str.strip()

In [22]:
print(df["Sex"].unique())

<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str


In [23]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,S


## Outlier Detection Using IQR Method

In [24]:
Q1 = df["Age"].quantile(0.25)
Q3 = df["Age"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower Limit:", lower)
print("Upper Limit:", upper)

Lower Limit: 2.5
Upper Limit: 54.5


In [25]:
outliers = df[(df["Age"] < lower) | (df["Age"] > upper)]

print("Number of Outliers:", len(outliers))
outliers.head()

Number of Outliers: 66


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
7,8,0,3,"Palsson, Master. Gosta Leonard",Male,2.0,3,1,349909,21.075,S
11,12,1,1,"Bonnell, Miss. Elizabeth",Female,58.0,0,0,113783,26.550,S
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",Female,55.0,0,0,248706,16.000,S
16,17,0,3,"Rice, Master. Eugene",Male,2.0,4,1,382652,29.125,Q
33,34,0,2,"Wheadon, Mr. Edward H",Male,66.0,0,0,C.A. 24579,10.500,S


In [26]:
df["Age"] = np.where(df["Age"] > upper, upper, df["Age"])
df["Age"] = np.where(df["Age"] < lower, lower, df["Age"])

In [27]:
df["Age"].describe()

count    891.000000
mean      29.039282
std       12.072074
min        2.500000
25%       22.000000
50%       28.000000
75%       35.000000
max       54.500000
Name: Age, dtype: float64

## Data Type Correction

In [28]:
df["PassengerId"] = df["PassengerId"].astype(str)
df["Age"] = df["Age"].astype(float)
df["Fare"] = df["Fare"].astype(float)

print(df.dtypes)

PassengerId        str
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Embarked           str
dtype: object


PassengerId was converted to string because it is an identifier and should not be treated as a numeric value. Age and Fare were kept as numeric (float) for analysis.

## Before vs After Summary

In [29]:
before_rows = 891
before_nulls = 866
before_duplicates = 0

after_rows = len(df)
after_nulls = df.isnull().sum().sum()
after_duplicates = df.duplicated().sum()

summary = pd.DataFrame({
    "Before": [before_rows, before_nulls, before_duplicates],
    "After": [after_rows, after_nulls, after_duplicates]
},
index=["Rows", "Null Values", "Duplicate Rows"])

summary

,Before,After
Rows,891,891
Null Values,866,0
Duplicate Rows,0,0


## Save Cleaned Dataset

In [30]:
df.to_csv("cleaned_dataset.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [31]:
import os

print(os.getcwd())
print(os.listdir())

/Users/luckysingh/Desktop/data cleaning
['cleaned_dataset.csv', 'cleaned_dataset.csv ', 'notebook.ipynb', 'requirements.txt       ', 'data']
